# Bengali LLM — Colab Inference Server
## Built with Unsloth · FastAPI · ngrok
---
**Run cells 1 → 7 in order.**

You only need to update two things before running:

**Cell 3** → set `ADAPTER_PATH` to your LoRA adapter folder on Google Drive

**Cell 7** → set `NGROK_AUTH_TOKEN` and `NGROK_STATIC_DOMAIN` from your ngrok dashboard

In [1]:
# ── Cell 1: Install dependencies ────────────────────────────
# Unsloth bundles compatible versions of transformers/peft/bitsandbytes.
# Install it first so there are no version conflicts.

# ── Cell 1: Install dependencies ────────────────────────────
!pip install -q unsloth
!pip install -q fastapi==0.124.1 starlette==0.49.1 uvicorn==0.34.0 pyngrok==7.2.2 nest_asyncio==1.6.0


from google.colab import drive
drive.mount('/content/drive')

print('✅ Dependencies installed and Drive mounted.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Dependencies installed and Drive mounted.


In [2]:
# ── Cell 2: Imports ─────────────────────────────────────────
import torch
import nest_asyncio
import uvicorn
import threading
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Dict, Optional
from unsloth import FastLanguageModel
from peft import PeftModel

nest_asyncio.apply()  # lets uvicorn run inside Colab's event loop
print('\u2705 Imports done.')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Imports done.


In [3]:
# ── Cell 3: Load Model ──────────────────────────────────────

# Base model — the same one used during fine-tuning
BASE_MODEL_ID = 'unsloth/Qwen2.5-3B-Instruct'

# ↓ UPDATE THIS: path to your LoRA adapter folder on Google Drive
ADAPTER_PATH  = '/content/drive/MyDrive/qwen_additional_qa_lora'

# Must match max_seq_length used during fine-tuning
MAX_SEQ_LENGTH = 2048

print('Loading base model with Unsloth (4-bit QLoRA)...')

# Step 1: Load base model via Unsloth — applies the same internal patches
# as during training, so the LoRA adapter weights will match exactly.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,    # auto: float16 on T4, bfloat16 on A100+
    load_in_4bit   = True,    # 4-bit quantisation — same as training
)

# Step 2: Attach your fine-tuned LoRA adapter
# Unsloth saves in standard PEFT format, so PeftModel.from_pretrained works.
print('Loading LoRA adapter from Drive...')
model = PeftModel.from_pretrained(model, ADAPTER_PATH)

# Step 3: Enable Unsloth's optimised inference kernel (~2x faster generation)
FastLanguageModel.for_inference(model)

print('\u2705 Model ready! Device:', next(model.parameters()).device)

Loading base model with Unsloth (4-bit QLoRA)...
==((====))==  Unsloth 2026.4.5: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Loading LoRA adapter from Drive...
✅ Model ready! Device: cuda:0


In [4]:
# from google.colab import drive
# drive.mount('/content/drive')

In [5]:
# ── Cell 4: System Prompt ───────────────────────────────────
# Prepended as the system message before every conversation.
# Edit to match the prompt style used during fine-tuning.

SYSTEM_PROMPT = (
    'You are a helpful Bengali language assistant developed by the '
    'Calcutta University Data Science Lab. You are fine-tuned on Bengali '
    'datasets covering reading comprehension, government schemes, history, '
    'technology, and everyday Bengali conversation. '
    'Always respond in the same language as the user\'s question. '
    'Be concise, factual, and helpful.'
)
print('\u2705 System prompt set.')

✅ System prompt set.


In [6]:
# ── Cell 5: Inference Function ──────────────────────────────

def run_inference(
    question: str,
    conversation_history: List[Dict[str, str]],
    max_new_tokens: int = 512,
    temperature: float = 0.3,
    top_p: float = 0.7,
) -> str:
    """
    Builds the Qwen2.5 chat template prompt, runs inference via Unsloth,
    and returns ONLY the newly generated assistant reply.

    Args:
        question             : current user message
        conversation_history : list of {role, content} for prior turns
                               (sent automatically by the LangGraph backend)
    """
    # 1. Build messages: system -> prior history -> new question
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    messages.extend(conversation_history)
    messages.append({'role': 'user', 'content': question})

    # 2. Apply Qwen2.5 chat template
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # 3. Tokenise and move to model device
    inputs = tokenizer([prompt_text], return_tensors='pt').to(model.device)
    input_length = inputs['input_ids'].shape[1]

    # 4. Generate — Unsloth's for_inference() speeds this up significantly
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = temperature,
            top_p              = top_p,
            do_sample          = True,
            repetition_penalty = 1.1,  # prevents repetition loops
            pad_token_id       = tokenizer.eos_token_id,
        )

    # 5. Decode only NEW tokens — strip the input prompt from output
    new_tokens = output_ids[0][input_length:]
    response   = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return response


# --- Optional quick sanity test (uncomment to run before starting server) ---
# test = run_inference('বাংলাদেশের রাজধানী কী?', [])
# print('Test reply:', test)

print('\u2705 Inference function defined.')

✅ Inference function defined.


In [7]:
# ── Cell 6: FastAPI App ─────────────────────────────────────

app = FastAPI(title='Bengali LLM Colab Inference')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['*'],
    allow_headers=['*'],
)


class InferenceRequest(BaseModel):
    question: str
    conversation_history: Optional[List[Dict[str, str]]] = []
    max_new_tokens: Optional[int] = 512
    temperature: Optional[float] = 0.7


class InferenceResponse(BaseModel):
    response: str


@app.get('/health')
def health():
    """Liveness check — polled by the LangGraph backend on startup."""
    return {'status': 'ok', 'model': 'Qwen2.5-3B Bengali Fine-tuned (Unsloth)'}


@app.post('/generate', response_model=InferenceResponse)
def generate(req: InferenceRequest):
    """
    Main inference endpoint called by the LangGraph backend.
    Receives: { question, conversation_history, max_new_tokens, temperature }
    Returns:  { response: '...' }
    """
    try:
        result = run_inference(
            question             = req.question,
            conversation_history = req.conversation_history or [],
            max_new_tokens       = req.max_new_tokens or 512,
            temperature          = req.temperature or 0.7,
        )
        return InferenceResponse(response=result)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


print('\u2705 FastAPI app defined.  Endpoints: GET /health   POST /generate')

✅ FastAPI app defined.  Endpoints: GET /health   POST /generate


In [8]:
# ── Cell 7: Start ngrok + uvicorn ───────────────────────────
import requests, threading, time, asyncio
from pyngrok import ngrok, conf
import uvicorn

NGROK_AUTH_TOKEN    = '3CP20LcKGnD9P3iN01nAjCiVP3l_3B3VTfC7FYrNs6aqk6wSR'
NGROK_STATIC_DOMAIN = 'rigid-bungee-jolt.ngrok-free.dev'
RENDER_BACKEND      = 'https://bengali-llm-backend.onrender.com'

# ─── Step 1: Kill old tunnels FIRST ──────────────────────────
print("Cleaning up old ngrok tunnels...")
ngrok.kill()
time.sleep(1)

# ─── Step 2: Connect fresh tunnel ────────────────────────────
conf.get_default().auth_token = NGROK_AUTH_TOKEN
listener = ngrok.connect(addr=8001, domain=NGROK_STATIC_DOMAIN)
public_url = listener.public_url

print(f'✅ Public URL  : {public_url}')
print(f'   /generate  : {public_url}/generate')
print(f'   /health    : {public_url}/health')

# ─── Step 3: Auto-update Render ──────────────────────────────
try:
    res = requests.post(
        f'{RENDER_BACKEND}/api/config/endpoint',
        json={'endpoint': f'{public_url}/generate'},
        timeout=10
    )
    print('✅ Render backend auto-updated!' if res.status_code == 200
          else f'⚠️ Render update failed: {res.status_code}')
except Exception as e:
    print(f'⚠️ Could not reach Render: {e}')

# ─── Step 4: Start uvicorn with a fresh event loop ───────────
def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host='0.0.0.0', port=8001, log_level='warning')
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)  # wait for uvicorn to actually start
print('🚀 Inference server is RUNNING!')


Cleaning up old ngrok tunnels...
✅ Public URL  : https://rigid-bungee-jolt.ngrok-free.dev
   /generate  : https://rigid-bungee-jolt.ngrok-free.dev/generate
   /health    : https://rigid-bungee-jolt.ngrok-free.dev/health
✅ Render backend auto-updated!
🚀 Inference server is RUNNING!


In [9]:
import requests
requests.post(
    "https://bengali-llm-backend.onrender.com/api/config/endpoint",
    json={"endpoint": "https://rigid-bungee-jolt.ngrok-free.app/generate"}
)


<Response [200]>